In [1]:
import sys
from pathlib import Path

ROOT = Path.cwd().parent

if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

In [2]:
import pandas as pd
import pickle
import torch
import os
import torch.nn.functional as F
import config

from tqdm.auto import tqdm

from torch.utils.data import  DataLoader
from src.metric import *
from sentence_transformers import SentenceTransformer,util,CrossEncoder


from src.bi_encoder_training import get_resume_embedding,get_jd_embedding,cosent_loss,compute_batch_embeddings
from src.cross_encoder_training import compute_batch_scores
from src.datasets import ResumeJDDataset



In [3]:
from torch.amp import autocast, GradScaler

scaler = GradScaler("cuda")
torch.set_float32_matmul_precision("high")

In [4]:
path=config.CLEANED_DATA_DIR

In [5]:
np.random.seed(config.SEED)
torch.manual_seed(config.SEED)
torch.cuda.manual_seed_all(config.SEED)

In [6]:
with open(os.path.join(path,'train_df.pkl'),'rb') as f:
    train_df=pickle.load(f)
    
with open(os.path.join(path,'val_df.pkl'),'rb') as f:
    val_df=pickle.load(f)
        
with open(os.path.join(path,'test_df.pkl'),'rb') as f:
    test_df=pickle.load(f)
    

In [7]:
BATCH_SIZE=config.CHUNK_BATCH_SIZE

In [10]:
bi_encoder = SentenceTransformer(os.path.join(config.CHUNKED_MODEL_DIR,'bi_encoder_chunked'))

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

In [11]:
unique_resume=train_df['resume_text'].drop_duplicates().tolist()

bi_encoder.eval()

resume_embs=[]

with torch.no_grad():
    for resume in tqdm(unique_resume):
        emb=get_resume_embedding(bi_encoder,resume,None)
        resume_embs.append(emb)
        del emb

pos_rows=train_df[train_df['label']==2]

  0%|          | 0/642 [00:00<?, ?it/s]

The `tokenize` method is deprecated, please use `preprocess` instead.


In [12]:
hard_neg_rows=[]
top_k_hard_neg=20

for _,row in pos_rows.iterrows():
    jd=row['job_description_text']
    pos_resume=row['resume_text']

    jd_emb=get_jd_embedding(bi_encoder,jd)

    hits=util.semantic_search(jd_emb,resume_embs,top_k=top_k_hard_neg)
    hits=hits[0]

    added=0
    for hit in hits:
        idx=hit['corpus_id']
        candidate_resume=unique_resume[idx]
        if candidate_resume==pos_resume:
            continue
            
        existing=train_df[(train_df["job_description_text"] == jd)
            &
            (train_df["resume_text"] == candidate_resume)]
        
        if len(existing)>0 and existing.iloc[0]['label']>0:
            continue

        hard_neg_rows.append({
            "job_description_text": jd,
            "resume_text": candidate_resume,
            "label": 0
        })

        added+=1
        if added>3:
            break

hard_neg_df=pd.DataFrame(hard_neg_rows)
print("Hard Negative Rows:", len(hard_neg_df))

Hard Negative Rows: 6168


In [14]:
enhanced_train=pd.concat([train_df,hard_neg_df])
print(len(enhanced_train))

12408


In [15]:
enhanced_train_path=os.path.join(config.DATA_DIR,'enhanced_train_df.pkl')
enhanced_train.to_pickle(enhanced_train_path)

In [12]:
#Retrain bi-encoder with enhanced data 

In [13]:
g = torch.Generator()
g.manual_seed(config.SEED)

In [14]:
labels=[float(config.label_to_score[label]) for label in enhanced_train['label']]
train_dataset=ResumeJDDataset(enhanced_train['resume_text'].values,enhanced_train['job_description_text'].values,labels)

bi_dataloader=DataLoader(train_dataset,batch_size=BATCH_SIZE,shuffle=True)

In [15]:
val_labels=[float(config.label_to_score[label]) for label in val_df['label']]
val_train_dataset=ResumeJDDataset( val_df['resume_text'].values,val_df['job_description_text'].values,val_labels)

val_bi_dataloader=DataLoader(val_train_dataset,batch_size=BATCH_SIZE,shuffle=False)

In [16]:
min_delta=0.01
count=0
best_score=float('-inf')
epochs=5
patience=2

best_bi_encoder_path=os.path.join(config.HD_MODEL_DIR,'bi_encoder_hard_negatives')
os.makedirs(config.HD_MODEL_DIR,exist_ok=True)

In [17]:
resume_chunk_map,jd_chunk_map={},{}

In [18]:
optimizer = torch.optim.AdamW(bi_encoder.parameters(), lr=2e-5)

In [19]:
for epoch in range(epochs):
    
    print(f"Epoch {epoch+1}/{epochs}")
    bi_encoder.train()
    
    total_loss=0
    
    progress_bar = tqdm(bi_dataloader, desc="Training")
    
    for resumes,jds,labels in progress_bar:
        optimizer.zero_grad()
        
        with autocast(device_type="cuda", dtype=torch.float16):
            resume_embs, jd_embs=compute_batch_embeddings(bi_encoder,resumes,jds,resume_chunk_map,jd_chunk_map)
            
            labels=labels.to(config.device)
            loss=cosent_loss(resume_embs,jd_embs,labels)
        
        scaler.scale(loss).backward()
        scaler.step(optimizer)
        scaler.update()
        
        total_loss += loss.item()
        
        progress_bar.set_postfix(loss=f"{loss.item():.4f}")
        
    print("Average Training Loss:",total_loss/len(progress_bar))
        
        
    #Validation    
    bi_encoder.eval()
    
    val_loss=0
    scores=[]
    with torch.no_grad():
        val_progress_bar = tqdm(val_bi_dataloader, desc="Validation")
        for resumes,jds,labels in val_progress_bar:
            
            with autocast(device_type="cuda", dtype=torch.float16):
                
                val_resume_embs, val_jd_embs=compute_batch_embeddings(bi_encoder,resumes,jds,resume_chunk_map,jd_chunk_map)
                
                labels=labels.to(config.device)
                loss=cosent_loss(val_resume_embs,val_jd_embs,labels)
                val_scores=F.cosine_similarity(val_resume_embs,val_jd_embs,dim=1)
                val_scores=val_scores.cpu().numpy()
                
            val_loss += loss.item()
            scores.extend(val_scores)
            val_progress_bar.set_postfix(loss=f"{loss.item():.4f}")
            
        print("Average Validation Loss:",val_loss/len(val_progress_bar))
        metrics=model_evaluation(scores,val_df,"job_description_text")
        print("NDCG:", metrics["ndcg_val"])
        print("MAP:", metrics["map_score"])
        
        final_score=0.6*metrics["ndcg_val"]+0.3*metrics["map_score"]+0.1*metrics["mrr_score"]
        
    if final_score>best_score+min_delta:
        best_score=final_score
        bi_encoder.save(best_bi_encoder_path)
        count=0
    else:
        count+=1

    if count==patience:
        print("Early stopping triggered.")
        break
                
            
    

Epoch 1/5


Training:   0%|          | 0/1551 [00:00<?, ?it/s]

Average Training Loss: 1.431853896022554


Validation:   0%|          | 0/146 [00:00<?, ?it/s]

Average Validation Loss: 0.03450183100896339
NDCG: 0.8596076933202359
MAP: 0.9651661604533608


...

Average Validation Loss: 0.02941499752541111
NDCG: 0.8564098612967258
MAP: 0.9605517905446975
Epoch 3/5


Training:   0%|          | 0/1551 [00:00<?, ?it/s]

Average Training Loss: 0.7080343832757395


Validation:   0%|          | 0/146 [00:00<?, ?it/s]

Average Validation Loss: 0.024279640961999763
NDCG: 0.8677705677249982
MAP: 0.9743385684619364
Early stopping triggered.


In [20]:
bi_encoder=SentenceTransformer(best_bi_encoder_path,device=config.device)

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

In [21]:
bi_encoder.eval()

with torch.no_grad():
    with autocast(device_type="cuda", dtype=torch.float16):
        val_resume_embs,val_jd_embs=compute_batch_embeddings(bi_encoder,test_df['resume_text'].values,
                                                            test_df['job_description_text'].values,resume_chunk_map,jd_chunk_map)
        
        scores=F.cosine_similarity(val_resume_embs,val_jd_embs,dim=1)
        scores=scores.cpu().numpy()
    
    metrics=model_evaluation(scores,test_df,"job_description_text")

In [22]:
print("Spearman:",metrics['spearman_score'])
print("Top-3 Accuracy:",metrics['topk_score'])
print("NDCG:",metrics['ndcg_val'])
print("MAP:",metrics['map_score'])
print("MRR:",metrics['mrr_score'])

Spearman: 0.430211692932492
Top-3 Accuracy: 0.8928571428571429
NDCG: 0.6442548167664551
MAP: 0.7864090108330589
MRR: 0.8098675914249684


Cross Encoder

In [16]:
cross_encoder = CrossEncoder(os.path.join(config.CHUNKED_MODEL_DIR, 'cross_encoder_chunked'),num_labels=1,device=config.device)

Loading weights:   0%|          | 0/105 [00:00<?, ?it/s]

In [17]:
labels=[float(config.label_to_score[label]) for label in enhanced_train['label']]
train_dataset=ResumeJDDataset(enhanced_train['resume_text'].values,enhanced_train['job_description_text'].values,labels)

train_loader=DataLoader(train_dataset,batch_size=BATCH_SIZE,shuffle=True)

In [18]:
cross_enoder_best_model_path=os.path.join(config.HD_MODEL_DIR,'cross_encoder_hard_negatives')
os.makedirs(config.HD_MODEL_DIR,exist_ok=True)

In [19]:
optimizer = torch.optim.AdamW(cross_encoder.parameters(), lr=2e-5)
loss_fn = torch.nn.MSELoss()

In [21]:
min_delta=0.01
count=0
best_score=float('-inf')
epochs=3
patience=2

In [22]:
resume_chunk_map,jd_chunk_map={},{}

In [23]:
for epoch in range(epochs):
    
    print(f"Epoch {epoch+1}/{epochs}")
    cross_encoder.train()
    
    total_loss=0
    
    progress_bar = tqdm(train_loader, desc="Training")
    
    for resumes,jds,labels in progress_bar:
        optimizer.zero_grad()
        with autocast(device_type="cuda", dtype=torch.float16):
            scores=compute_batch_scores(cross_encoder,resumes,jds,resume_chunk_map,jd_chunk_map)
            labels=labels.to(config.device,dtype=scores.dtype)
            
            loss=loss_fn(scores,labels)
        
        scaler.scale(loss).backward()
        scaler.step(optimizer)
        scaler.update()
        
        total_loss+=loss.item()
        
        progress_bar.set_postfix(loss=f"{loss.item():.4f}")
        
    average_loss=total_loss/len(progress_bar)
    print("Average Training Loss:",average_loss)
        
        
    #Validation    
    cross_encoder.eval()
    
    with torch.no_grad():
        with autocast(device_type="cuda", dtype=torch.float16):
            val_scores=compute_batch_scores(cross_encoder,val_df['resume_text'].values,
                                            val_df['job_description_text'].values,resume_chunk_map,jd_chunk_map)
            val_scores=val_scores.cpu().numpy()
       
        metrics=model_evaluation(val_scores,val_df,"job_description_text")
        print("NDCG:", metrics["ndcg_val"])
        print("MAP:", metrics["map_score"])
        
        final_score=0.6*metrics["ndcg_val"]+0.3*metrics["map_score"]+0.1*metrics["mrr_score"]
        
    if final_score>best_score+min_delta:
        best_score=final_score
        cross_encoder.save(cross_enoder_best_model_path)
        count=0
    else:
        count+=1

    if count==patience:
        print("Early stopping triggered.")
        break
                
            
    

Epoch 1/3


Training:   0%|          | 0/1551 [00:00<?, ?it/s]

Average Training Loss: 0.10902891728890728
NDCG: 0.7088592222649475
MAP: 0.7844969860183387


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Epoch 2/3


...

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Epoch 3/3


Training:   0%|          | 0/1551 [00:00<?, ?it/s]

Average Training Loss: 0.07305223269098055
NDCG: 0.7570220298544816
MAP: 0.8268214927916122


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

In [24]:
cross_encoder=CrossEncoder(cross_enoder_best_model_path,num_labels=1,device=config.device)

Loading weights:   0%|          | 0/105 [00:00<?, ?it/s]

In [25]:
resume_chunk_map,jd_chunk_map={},{}

In [26]:
cross_encoder.eval()
with torch.no_grad():
    scores=compute_batch_scores(cross_encoder,test_df['resume_text'].values,test_df['job_description_text'].values)
    scores=scores.cpu().numpy()
    metrics=model_evaluation(scores,test_df,"job_description_text")

In [27]:
print("Spearman:",metrics['spearman_score'])
print("Top-3 Accuracy:",metrics['topk_score'])
print("NDCG:",metrics['ndcg_val'])
print("MAP:",metrics['map_score'])
print("MRR:",metrics['mrr_score'])

Spearman: 0.3337770709976757
Top-3 Accuracy: 0.9285714285714286
NDCG: 0.6580633964859436
MAP: 0.7348464659942847
MRR: 0.8494850777637662
